# Virtualizing TIFF from a COG in Local using Virtual TIFF

I'm following the tutorial here: https://virtualizarr.readthedocs.io/en/stable/usage.html#usage

LocalStore: Local filesystem storage providing the same object store interface.

In [1]:
pip install obstore
pip install obspec-utils
pip install virtualizarr
pip install virtual-tiff
pip install --upgrade numpy
pip install rustac
pip install xstac

  Using cached obstore-0.9.2-cp311-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (3.9 kB)
Using cached obstore-0.9.2-cp311-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (4.2 MB)
Note: you may need to restart the kernel to use updated packages.


Now we're coming back to virtualizing, and now I need to blend these two tutorials:
1. https://virtualizarr.readthedocs.io/en/stable/usage.html#opening-files-as-virtual-datasets
2. https://virtual-tiff.readthedocs.io/en/latest/#background

In [9]:
import xarray as xr
from obstore.store import LocalStore
from obspec_utils.registry import ObjectStoreRegistry

from virtualizarr import open_virtual_dataset, open_virtual_mfdataset
# from virtualizarr.parsers import HDFParser
from virtual_tiff import VirtualTIFF

from pathlib import Path

In [10]:
store_path = Path.cwd()
file_path = str(store_path / "LC09_L2SP_046029_20230824_02_T1_RGB.tiff")
file_url = f"file://{file_path}"

store = LocalStore(prefix=store_path)
registry = ObjectStoreRegistry({file_url: store})

import warnings
warnings.filterwarnings(
  "ignore",
  message="Numcodecs codecs are not in the Zarr version 3 specification*",
  category=UserWarning
)

parser = VirtualTIFF(ifd=0)
manifest_store = parser(url=file_url, registry=registry)
ds = xr.open_zarr(manifest_store, zarr_format=3, consolidated=False)
ds.load()

<xarray.Dataset> Size: 740MB
Dimensions:  (band: 3, y: 7901, x: 7801)
Dimensions without coordinates: band, y, x
Data variables:
    0        (band, y, x) float32 740MB nan nan nan nan nan ... nan nan nan nan

Now try opening a virtual dataset:

In [11]:
ds = open_virtual_dataset(
    url=file_url,
    registry=registry,
    parser=VirtualTIFF(ifd=0)
)

These two tutorials create a STAC layer on icechunk store: 
1. https://nasa-impact.github.io/dse-virtual-zarr-workshop/examples/03_STAC_Collection_for_Virtual_Icechunk_Store.html
2. https://nasa-impact.github.io/dse-virtual-zarr-workshop/examples/04_STAC_Items_for_legacy_files_in_Virtual_Icechunk_Store.html

Following is a failed attempt to create a STAC layer on a virtualized TIFF:

In [25]:
import pystac
import rasterio
from shapely.geometry import box, mapping
from datetime import datetime
import re
import os

Set bounds from the file itself (this might need to be downloaded first, which is annoying)

In [36]:
with rasterio.open(file_path) as src:
    bounds = src.bounds
    bbox = [bounds.left, bounds.bottom, bounds.right, bounds.top]

The same is true for the datetime

In [37]:
dt = datetime.strptime(
    re.search(r"\d{8}", file_path).group(),
    "%Y%m%d"
)

The assets reference the virtual store:

In [38]:
asset = pystac.Asset(
    href=str(manifest_store),   #here it references the virtual dataset
    media_type="application/vnd+zarr",
    roles=["data", "virtual"]
)

item.add_asset("data", asset)

In [39]:
bbox

[391785.0, 4820385.0, 625815.0, 5057415.0]

In [40]:
item.to_dict()

{'type': 'Feature',
 'stac_version': '1.1.0',
 'stac_extensions': ['https://stac-extensions.github.io/projection/v2.0.0/schema.json',
  'https://stac-extensions.github.io/eo/v1.1.0/schema.json'],
 'id': 'LC09_L2SP_046029_20230824_02_T1_RGB',
 'geometry': {'type': 'Polygon',
  'coordinates': (((625815.0, 4820385.0),
    (625815.0, 5057415.0),
    (391785.0, 5057415.0),
    (391785.0, 4820385.0),
    (625815.0, 4820385.0)),)},
 'bbox': [391785.0, 4820385.0, 625815.0, 5057415.0],
 'properties': {'proj:code': 'EPSG:32610',
  'proj:shape': [7901, 7801],
  'proj:transform': [30.0, 0.0, 391785.0, 0.0, -30.0, 5057415.0],
  'proj:bbox': [391785.0, 4820385.0, 625815.0, 5057415.0],
  'eo:bands': [{'name': 'red', 'common_name': 'red'},
   {'name': 'green', 'common_name': 'green'},
   {'name': 'blue', 'common_name': 'blue'}],
  'datetime': '2023-08-24T00:00:00Z'},
 'links': [],
 'assets': {'data': {'href': "ManifestStore(group=\nManifestGroup(\n    arrays={'0': ManifestArray<shape=(3, 7901, 7801), 

To complete this, I need to create some saved artifact from the virtual store. Is this the way to go? Spending some subsequent time thinking about the most generalizable pathway for our needs in this.